# Example 10.4 / 2: Reformulate a Min-Max Objective

Companion notebook for **10.4 Piecewise-Linear Terms** in the AMPL lab Documentation, section "Reformulate a Min-Max Objective". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Reformulate a Min-Max Objective

The fairest assignment minimizes the largest cost charged to any one person:

$$\min_{Assign}\; \max_{i \in PEOPLE} \sum_{j \in PROJECTS} cost_{ij}\, Assign_{ij}.$$

This function is pieced together from linear functions, but each piece depends on several
variables, so it is not separable and cannot be written with `<<...>>`. Introduce a variable `M`
that must be at least every person's cost and minimize it; at the optimum `M` equals the largest
cost. A "max-min" objective uses `maximize` and `M <= ...` in the same way.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PEOPLE;  # people to assign
set PROJECTS;  # projects to staff
param supply {PEOPLE} >= 0;  # hours each person is available
param demand {PROJECTS} >= 0;  # hours each project requires
check: sum {i in PEOPLE} supply[i] = sum {j in PROJECTS} demand[j];  # balanced hours
param cost {PEOPLE,PROJECTS} >= 0;  # cost per hour of work
param limit {PEOPLE,PROJECTS} >= 0;  # most hours a person gives one project

var M;  # largest cost of any one person
var Assign {i in PEOPLE, j in PROJECTS} >= 0, <= limit[i,j];  # hours assigned

minimize Max_Cost: M;
subject to M_def {i in PEOPLE}:  # M is at least each person's cost
    M >= sum {j in PROJECTS} cost[i,j] * Assign[i,j];
subject to Supply {i in PEOPLE}: sum {j in PROJECTS} Assign[i,j] = supply[i];  # use all hours
subject to Demand {j in PROJECTS}: sum {i in PEOPLE} Assign[i,j] = demand[j];  # meet requirements

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PEOPLE"] = ["Ann", "Bob"]
ampl.set["PROJECTS"] = ["A", "B"]
ampl.param["supply"] = {"Ann": 6, "Bob": 4}
ampl.param["demand"] = {"A": 5, "B": 5}
ampl.param["cost"] = {("Ann", "A"): 2, ("Ann", "B"): 3, ("Bob", "A"): 4, ("Bob", "B"): 1}
ampl.param["limit"] = {(i, j): 6 for i in ["Ann", "Bob"] for j in ["A", "B"]}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.var["M"].value())
print(ampl.var["Assign"].to_dict())

HiGHS 1.14.0: optimal solution; objective 13
0 simplex iterations
0 barrier iterations
solved 13.0
{('Ann', 'A'): 5, ('Ann', 'B'): 1, ('Bob', 'A'): 0, ('Bob', 'B'): 4}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PEOPLE := Ann Bob;
set PROJECTS := A B;
param supply := Ann 6  Bob 4;
param demand := A 5  B 5;
param cost:  A  B :=
  Ann        2  3
  Bob        4  1 ;
param limit default 6;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Assign, M;
display {i in PEOPLE} sum {j in PROJECTS} cost[i,j] * Assign[i,j];

HiGHS 1.14.0: optimal solution; objective 13
0 simplex iterations
0 barrier iterations
Assign :=
Ann A   5
Ann B   1
Bob A   0
Bob B   4
;

M = 13

sum{j in PROJECTS} cost[i,j]*Assign[i,j] [*] :=
Ann  13
Bob   4
;

